In [ ]:
import base64,hashlib,io,pathlib,subprocess,sys,zipfile,torch
assert torch.cuda.is_available(),"CUDA allocation required"
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='2396b21e9e73b2fc45f82490ac01975b4a9367a5fe2f1f1dbd9307f9d29af61f'
root=pathlib.Path("/kaggle/working/t3d-rigged-export");root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:z.extractall(root)
subprocess.run([sys.executable,"-m","pip","install","hilbertcurve>=2,<3","trimesh>=4,<5","Pillow>=10","opencv-python-headless","ninja"],check=True)
logs=[]
for pattern in ("test_articulation.py","test_import_rig.py","test_export.py","test_glb.py","test_materials.py","test_metrics.py","test_skin_ae.py"):
  r=subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",pattern,"-v"],cwd=root,capture_output=True,text=True)
  logs.append(r.stdout+r.stderr);print(logs[-1]);assert r.returncode==0
(root.parent/"rigged-export-tests.log").write_text("\n".join(logs))
fixture=pathlib.Path("/kaggle/input/transformer-native-3d-textured-rig-inputs")
subprocess.run([sys.executable,"-m","tools.probe_export","--source-sha256",'2396b21e9e73b2fc45f82490ac01975b4a9367a5fe2f1f1dbd9307f9d29af61f',"--skinae-checkpoint",str(root/"artifacts/kaggle-skin-position-ablation-v3/loss-1.0/skin-pose-pilot-holdout-CesiumMan.pt"),"--real-rig-source",str(fixture/"CesiumMan.glb"),"--real-grid-size","128","--real-texture-size","2048","--dependency-cache","/kaggle/input/transformer-native-3d-cuda-wheels-cu130"],cwd=root,check=True)
